# COMP 4930SEF Project

**Group number:** 43

| Member | Full name | 8-digit student ID |
| --- | --- | --- |
| M1 | Chiang Sui Kit | 13492598 |
| M2 | Chen Hill Fung | 13492120 |
| M3 | To be supplied | 14108542 |

## Introduction

We will compare three classifiers on the CIFAR-100 animals-and-people subset, adapt the selected model to a dataset photographed and labelled by our group, and develop a text classifier for AG News. Each task will document the data, methodology, implementation, results and interpretation.

## Environment and shared utilities

Start JupyterLab from the project root using `mise run lab`. The following cell records the execution environment. Model experiments and their shared utilities will be added task by task.

In [1]:
import sys
from pathlib import Path

import tensorflow as tf

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "mise.toml").is_file():
    raise RuntimeError("Start JupyterLab from the project root with mise run lab.")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
SEED = 42
tf.keras.utils.set_random_seed(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {tf.keras.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

Python: 3.12.15
TensorFlow: 2.21.0
Keras: 3.15.1
GPUs: []


## Task 1 — CIFAR-100 animals and people

We compare ImageNet transfer at a common input resolution against a smaller CNN trained directly on CIFAR pixels. The shared validation split measures performance on unseen examples of the same 50 classes. It also lets us examine whether the extra parameters and training time of a pretrained network help on these small images.

Choose a run mode before launching Jupyter from the project root. The default `preview` loads the definitions. `smoke` runs all three models on a small balanced subset. `train` runs the full validation comparison on the GPU. After reviewing those results, `final` uses the saved checkpoints and evaluates the selected model. Save any code edits before running. [Legion WSL2 instructions](docs/task1-training.md)

In [2]:
import hashlib
import json
import os
from collections.abc import Iterator
from contextlib import contextmanager
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

MODE = os.environ.get("TASK1_MODE", "preview").lower()
if MODE not in {"preview", "smoke", "train", "final"}:
    raise ValueError("TASK1_MODE must be preview, smoke, train, or final.")
BATCH_SIZE = int(os.environ.get("TASK1_BATCH_SIZE", "32"))
if BATCH_SIZE < 1:
    raise ValueError("TASK1_BATCH_SIZE must be at least 1.")
TASK1_DIR = Path("artifacts/task1")
RUN_SEEDS = (42, 43, 44)
gpus = tf.config.list_physical_devices("GPU")
if MODE in {"train", "final"} and not gpus:
    raise RuntimeError(
        "The full experiment needs the Legion GPU. See docs/task1-training.md."
    )

initial_policy = tf.keras.mixed_precision.global_policy().name
device_name = "CPU"
if MODE != "preview":
    if gpus:
        for gpu in gpus:
            if not tf.config.experimental.get_memory_growth(gpu):
                tf.config.experimental.set_memory_growth(gpu, True)
        with tf.device("/GPU:0"):
            probe = tf.linalg.matmul(tf.ones((2, 2)), tf.ones((2, 2)))
        if "GPU:0" not in probe.device or float(probe[0, 0]) != 2.0:
            raise RuntimeError("The GPU matrix multiplication check failed.")
        device_name = tf.config.experimental.get_device_details(gpus[0]).get(
            "device_name", "GPU"
        )
    tf.config.experimental.enable_op_determinism()
    tf.keras.mixed_precision.set_global_policy("mixed_float16" if gpus else "float32")
print(f"Task 1 mode = {MODE}, batch size = {BATCH_SIZE}")
if MODE != "preview":
    print(
        f"Device = {device_name}, precision = {tf.keras.mixed_precision.global_policy().name}"
    )

Task 1 mode = smoke, batch size = 32
Device = CPU, precision = float32


### Data and evaluation design

The appendix fixes the 50 fine-label IDs for this experiment. The table in code pairs those IDs with the names shipped with CIFAR-100. In particular, ID 57 is **pear**. We keep that class so the experiment uses the specified subset. Each example is a 32 × 32 RGB image. The split seed stays at 42 for all model seeds. Each class contributes 400 training images, 100 validation images and 100 images in the official test set. Only the selected model reaches the final test evaluation. [CIFAR-100 dataset](https://www.cs.toronto.edu/~kriz/cifar.html) · [Keras loader](https://keras.io/api/datasets/cifar100/) · [Fine-label names in source order](https://github.com/tensorflow/datasets/blob/master/tensorflow_datasets/testing/metadata/cifar100/3.0.2/label.labels.txt)

In [3]:
FINE_LABELS = {
    1: "aquarium_fish",
    2: "baby",
    3: "bear",
    4: "beaver",
    6: "bee",
    7: "beetle",
    11: "boy",
    14: "butterfly",
    15: "camel",
    18: "caterpillar",
    19: "cattle",
    21: "chimpanzee",
    24: "cockroach",
    26: "crab",
    27: "crocodile",
    29: "dinosaur",
    30: "dolphin",
    31: "elephant",
    32: "flatfish",
    34: "fox",
    35: "girl",
    36: "hamster",
    38: "kangaroo",
    42: "leopard",
    43: "lion",
    44: "lizard",
    45: "lobster",
    46: "man",
    50: "mouse",
    55: "otter",
    57: "pear",
    63: "porcupine",
    64: "possum",
    65: "rabbit",
    66: "raccoon",
    67: "ray",
    72: "seal",
    73: "shark",
    74: "shrew",
    75: "skunk",
    77: "snail",
    78: "snake",
    79: "spider",
    80: "squirrel",
    91: "trout",
    93: "turtle",
    95: "whale",
    97: "wolf",
    98: "woman",
    99: "worm",
}
FINE_IDS = tuple(FINE_LABELS)
CLASS_NAMES = tuple(FINE_LABELS.values())


def load_task1_data():
    """Return three native-resolution splits and a digest of the development row IDs."""
    (dev_x, dev_y), (test_x, test_y) = tf.keras.datasets.cifar100.load_data(
        label_mode="fine"
    )
    to_subset = np.full(100, -1, dtype=np.int32)
    to_subset[list(FINE_IDS)] = np.arange(len(FINE_IDS))
    dev_y = to_subset[dev_y.ravel()]
    test_y = to_subset[test_y.ravel()]
    dev_rows = np.flatnonzero(dev_y >= 0)
    test_rows = np.flatnonzero(test_y >= 0)
    fit_rows, val_rows = train_test_split(
        dev_rows, test_size=5_000, random_state=42, stratify=dev_y[dev_rows]
    )

    if np.intersect1d(fit_rows, val_rows).size:
        raise ValueError("Development partitions overlap.")
    train = (dev_x[fit_rows], dev_y[fit_rows])
    validation = (dev_x[val_rows], dev_y[val_rows])
    test = (test_x[test_rows], test_y[test_rows])
    for (images, labels), n in zip(
        (train, validation, test), (400, 100, 100), strict=True
    ):
        if (
            images.shape != (n * len(CLASS_NAMES), 32, 32, 3)
            or images.dtype != np.uint8
        ):
            raise ValueError(
                "Expected native 32 × 32 RGB images in channels-last order."
            )
        if not np.all(np.bincount(labels, minlength=len(CLASS_NAMES)) == n):
            raise ValueError(f"Expected {n} examples per fine class in this partition.")

    row_bytes = np.concatenate((fit_rows, val_rows)).astype("<i4").tobytes()
    return train, validation, test, hashlib.sha256(row_bytes).hexdigest()

### Reading model performance

Accuracy counts the images assigned the correct fine label. Top-5 accuracy also counts a correct label among the five highest scores. Macro-F1 gives every class the same weight and responds to both missed examples and incorrect assignments. The confusion matrix uses the true class on each row. A bright off-diagonal cell identifies a pair to inspect in the original images.

In [4]:
def classification_metrics(y_true: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    """Compute the same three metrics from one inference pass on each checkpoint."""
    y_true = np.asarray(y_true).ravel()
    scores = np.asarray(scores)
    if scores.shape != (len(y_true), len(CLASS_NAMES)) or not np.isfinite(scores).all():
        raise ValueError("Expected one finite score per image and fine class.")
    ranked = np.argsort(-scores, axis=1, kind="stable")
    predicted = ranked[:, 0]
    return {
        "accuracy": float(np.mean(predicted == y_true)),
        "macro_f1": float(
            f1_score(
                y_true,
                predicted,
                labels=np.arange(len(CLASS_NAMES)),
                average="macro",
                zero_division=0,
            )
        ),
        "top5_accuracy": float(np.any(ranked[:, :5] == y_true[:, None], axis=1).mean()),
    }


def show_class_errors(y_true: np.ndarray, scores: np.ndarray, title: str) -> None:
    """Show per-class metrics, common directional errors and their full matrix."""
    y_true = np.asarray(y_true).ravel()
    predicted = np.argmax(scores, axis=1)
    labels = np.arange(len(CLASS_NAMES))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        predicted,
        labels=labels,
        zero_division=0,
    )
    per_class = pd.DataFrame(
        {
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "support": support,
        },
        index=pd.Index(CLASS_NAMES, name="fine class"),
    )
    display(per_class.sort_values("F1").round(3))

    counts = confusion_matrix(y_true, predicted, labels=labels)
    errors = counts.copy()
    np.fill_diagonal(errors, 0)
    expected, assigned = np.nonzero(errors)
    pairs = pd.DataFrame(
        {
            "true": np.asarray(CLASS_NAMES)[expected],
            "predicted": np.asarray(CLASS_NAMES)[assigned],
            "images": errors[expected, assigned],
        }
    )
    display(pairs.nlargest(5, "images").reset_index(drop=True))

    fig, ax = plt.subplots(figsize=(11, 9), layout="constrained")
    fractions = np.divide(counts, np.maximum(counts.sum(axis=1, keepdims=True), 1))
    mesh = ax.imshow(fractions, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(labels, CLASS_NAMES, rotation=90, fontsize=7)
    ax.set_yticks(labels, CLASS_NAMES, fontsize=7)
    ax.set(xlabel="Predicted fine class", ylabel="True fine class", title=title)
    fig.colorbar(mesh, ax=ax, label="Fraction of each true class")
    plt.show()
    plt.close(fig)

### Three candidate models

| Candidate | Design | Question |
| --- | --- | --- |
| MobileNet V1 | ImageNet weights, depthwise and pointwise convolutions, 128 × 128 input | How well does a lightweight pretrained feature extractor transfer to these classes? |
| Custom CNN | Three blocks with 32, 64 and 128 channels, two 3 × 3 convolutions per block and 2 × 2 pooling after the first two | How much can a small model learn directly from the original 32 × 32 pixels? |
| EfficientNetV2B0 | ImageNet weights, fused and standard mobile inverted bottleneck blocks, 128 × 128 input | Does this alternative pretrained backbone improve recognition enough to justify its training cost? |

Each network averages the final feature map over its spatial dimensions, applies dropout at 0.25 and returns 50 float32 logits. The layer named `representation` exposes the pooled features for Task 2. Both transfer candidates resize each batch inside the model. MobileNet scales pixels to [−1, 1]. EfficientNetV2B0 applies its own preprocessing to pixels in [0, 255]. The custom model scales pixels to [0, 1]. Training adds horizontal flips and small translations of up to 6% of each dimension. At evaluation, Keras passes each image through the same resize and scaling steps with augmentation disabled.

Architecture and preprocessing references include [MobileNet V1](https://arxiv.org/abs/1704.04861), [EfficientNetV2](https://proceedings.mlr.press/v139/tan21a.html), the [Keras MobileNet API](https://keras.io/api/applications/mobilenet/) and the [Keras EfficientNetV2 API](https://www.tensorflow.org/api_docs/python/tf/keras/applications/EfficientNetV2B0).

In [5]:
CANDIDATES = ("mobilenet_v1", "custom_cnn", "efficientnetv2b0")


def build_candidate(name: str, seed: int, *, weights: str | None = "imagenet"):
    """Build a 50-way classifier accepting batches of raw RGB images."""
    if name not in CANDIDATES:
        raise ValueError(f"Unknown candidate: {name}")
    layers = tf.keras.layers
    size = 32 if name == "custom_cnn" else 128
    inputs = layers.Input(shape=(None, None, 3), dtype="uint8", name="rgb")
    x = layers.Resizing(size, size, dtype="float32", name="resize")(inputs)
    x = layers.RandomFlip("horizontal", seed=seed, dtype="float32")(x)
    x = layers.RandomTranslation(
        0.06, 0.06, fill_mode="reflect", seed=seed + 1, dtype="float32"
    )(x)
    if name == "custom_cnn":
        x = layers.Rescaling(1.0 / 255, dtype="float32")(x)
        for channels in (32, 64, 128):
            for _ in range(2):
                x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
                x = layers.BatchNormalization()(x)
                x = layers.Activation("relu")(x)
            if channels < 128:
                x = layers.MaxPooling2D()(x)
    else:
        if name == "mobilenet_v1":
            x = layers.Rescaling(1.0 / 127.5, offset=-1, dtype="float32")(x)
            backbone = tf.keras.applications.MobileNet(
                input_shape=(size, size, 3), include_top=False, weights=weights
            )
        else:
            backbone = tf.keras.applications.EfficientNetV2B0(
                input_shape=(size, size, 3),
                include_top=False,
                include_preprocessing=True,
                weights=weights,
            )
        backbone.trainable = False
        x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="representation", dtype="float32")(x)
    x = layers.Dropout(0.25, seed=seed + 2)(x)
    logits = layers.Dense(50, name="classifier", dtype="float32")(x)
    return tf.keras.Model(inputs, logits, name=name)


def unfreeze_final_blocks(model) -> None:
    """Release the last convolutional blocks while freezing BatchNorm statistics."""
    backbone = next(
        layer for layer in model.layers if isinstance(layer, tf.keras.Model)
    )
    prefixes = (
        ("conv_dw_12", "conv_pw_12", "conv_dw_13", "conv_pw_13")
        if model.name == "mobilenet_v1"
        else ("block6", "top_conv")
    )
    backbone.trainable = True
    for layer in backbone.layers:
        layer.trainable = layer.name.startswith(prefixes) and not isinstance(
            layer, tf.keras.layers.BatchNormalization
        )
    if not backbone.trainable_weights:
        raise ValueError(f"No trainable layers matched {prefixes} in {backbone.name}.")

### Training and checkpoint selection

The custom CNN starts from random weights and receives up to 18 epochs with Adam at 0.001. Each transfer candidate trains its head for up to four epochs at 0.001. Starting from the best head weights, it then fine-tunes the last MobileNet blocks (12 and 13) or EfficientNet stage 6 and its top convolution for up to eight epochs at 0.00001. The smaller step size lets the upper features adapt gradually. Frozen BatchNorm layers use their saved statistics throughout transfer training. With the pinned Keras version, EfficientNetV2B0 also applies its built-in stochastic depth during fitting. This randomly skips some residual contributions as a regulariser. All dropout is disabled for validation and test predictions. Keras recompiles with a fresh optimizer after the trainable layers change. [Keras transfer learning guide](https://keras.io/guides/transfer_learning/)

Each stage stops after three consecutive epochs without higher validation accuracy. The saved model comes from the highest validation accuracy across all stages, retaining the earliest epoch on a tie. The comparison includes differences in pretraining, resolution and training budget. Recorded fit time includes checkpoint writes and excludes initial weight downloads.

In [6]:
def train_candidate(
    name: str,
    seed: int,
    train_ds,
    val_ds,
    run_dir: Path,
    *,
    weights: str | None = "imagenet",
    head_epochs: int,
    tune_epochs: int,
    scratch_epochs: int,
):
    """Train one seed and keep the highest validation accuracy across its stages."""
    policy = tf.keras.mixed_precision.global_policy().name
    tf.keras.backend.clear_session()
    tf.keras.mixed_precision.set_global_policy(policy)
    tf.keras.utils.set_random_seed(seed)
    model = build_candidate(name, seed, weights=weights)
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = run_dir / "best.keras"
    stages = (
        [("scratch", scratch_epochs, 1e-3)]
        if name == "custom_cnn"
        else [("head", head_epochs, 1e-3), ("fine_tune", tune_epochs, 1e-5)]
    )
    history: list[dict[str, str | float | int]] = []
    best_epoch = 0
    best_stage = ""
    best_accuracy = -np.inf
    start = perf_counter()
    for stage, epochs, learning_rate in stages:
        if epochs < 1:
            raise ValueError(f"The {stage} stage needs at least one epoch.")
        if stage == "fine_tune":
            unfreeze_final_blocks(model)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
            jit_compile=False,
        )
        save_best = tf.keras.callbacks.ModelCheckpoint(
            str(checkpoint),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            initial_value_threshold=best_accuracy,
        )
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", mode="max", patience=3, restore_best_weights=True
        )
        print(f"{name} | seed {seed} | {stage} | learning rate {learning_rate:g}")
        fit = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            callbacks=[save_best, early_stop],
            shuffle=False,
            verbose=2,
        )
        for i in range(len(fit.epoch)):
            metrics = {
                key: float(fit.history[key][i])
                for key in ("loss", "accuracy", "val_loss", "val_accuracy")
            }
            if not all(np.isfinite(value) for value in metrics.values()):
                raise ValueError(f"Non-finite training metrics in {name}, seed {seed}.")
            history.append({"stage": stage, "epoch": len(history) + 1, **metrics})
            if metrics["val_accuracy"] > best_accuracy:
                best_accuracy = metrics["val_accuracy"]
                best_epoch = len(history)
                best_stage = stage

    return {
        "checkpoint": str(checkpoint),
        "history": history,
        "parameters": model.count_params(),
        "seconds": perf_counter() - start,
        "best_epoch": best_epoch,
        "best_stage": best_stage,
        "best_val_accuracy": best_accuracy,
    }

### Training and validation

Each seed starts with fresh weights and a freshly shuffled training stream. Validation images keep a fixed order. For each run we reload the checkpoint with the highest validation accuracy and calculate every validation metric from the same predictions. Resetting the seed before dataset construction fixes the shuffle stream even when earlier runs are loaded from disk. A completed run can be reused after restarting the kernel. The experiment directory records the split, software versions, precision, batch size and the code used to train it.

In [7]:
@contextmanager
def task1_writer() -> Iterator[None]:
    """Hold an OS file lock while one kernel writes run or final artifacts."""
    TASK1_DIR.mkdir(parents=True, exist_ok=True)
    with (TASK1_DIR / ".writer.lock").open("a+b") as handle:
        handle.seek(0)
        try:
            if sys.platform == "win32":
                import msvcrt

                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl

                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError as exc:
            raise RuntimeError(
                f"Could not lock {TASK1_DIR}. Check for another active kernel."
            ) from exc
        yield  # Closing the handle releases the lock, including on kernel exit.


def make_batches(pair, batch_size: int, seed: int | None = None):
    images, labels = pair
    batches = tf.data.Dataset.from_tensor_slices((images, labels))
    if seed is not None:
        batches = batches.shuffle(len(labels), seed=seed, reshuffle_each_iteration=True)
    return batches.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def smoke_sample(pair, per_class: int):
    images, labels = pair
    indices = np.concatenate(
        [np.flatnonzero(labels == i)[:per_class] for i in range(len(CLASS_NAMES))]
    )
    return images[indices], labels[indices]


def file_digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def write_json(path: Path, record: dict) -> None:
    temporary = path.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(record, indent=2, allow_nan=False), encoding="utf-8"
    )
    temporary.replace(path)


def experiment_settings(split_id: str) -> dict:
    """Identify reusable runs from the saved notebook's training cells and data."""
    report = json.loads(Path("project_report.ipynb").read_text(encoding="utf-8"))
    sources = [
        "".join(cell["source"])
        for cell in report["cells"]
        if "task1-protocol" in cell["metadata"].get("tags", [])
    ]
    return {
        "protocol_sha256": hashlib.sha256("\n".join(sources).encode()).hexdigest(),
        "split_sha256": split_id,
        "smoke": MODE == "smoke",
        "batch_size": BATCH_SIZE,
        "tensorflow": tf.__version__,
        "keras": tf.keras.__version__,
        "device": device_name,
        "precision": tf.keras.mixed_precision.global_policy().name,
        "fine_ids": list(FINE_IDS),
        "classes": list(CLASS_NAMES),
    }


def validation_run(name: str, seed: int, fit_pair, val_pair, directory: Path) -> dict:
    """Finish one fit, or read its completed checkpoint and validation predictions."""
    run_dir = directory / name / str(seed)
    run_dir.mkdir(parents=True, exist_ok=True)
    record_file = run_dir / "run.json"
    checkpoint = run_dir / "best.keras"
    predictions = run_dir / "validation.npy"
    if record_file.is_file():
        record = json.loads(record_file.read_text(encoding="utf-8"))
        if (
            file_digest(checkpoint) != record["checkpoint_sha256"]
            or file_digest(predictions) != record["validation_sha256"]
        ):
            raise ValueError(f"Saved run files do not match {record_file}.")
        print(f"Loaded {name}, seed {seed}")
        return record
    if MODE == "final":
        raise RuntimeError(
            f"Missing completed validation run {name}, seed {seed}. Run train first."
        )
    head, tune, scratch = (1, 1, 1) if MODE == "smoke" else (4, 8, 18)
    tf.keras.utils.set_random_seed(seed)
    record = train_candidate(
        name,
        seed,
        make_batches(fit_pair, BATCH_SIZE, seed),
        make_batches(val_pair, BATCH_SIZE),
        run_dir,
        head_epochs=head,
        tune_epochs=tune,
        scratch_epochs=scratch,
    )
    best_model = tf.keras.models.load_model(checkpoint, compile=False)
    scores = best_model.predict(val_pair[0], batch_size=BATCH_SIZE, verbose=0)
    metrics = classification_metrics(val_pair[1], scores)
    if not np.isclose(metrics["accuracy"], record["best_val_accuracy"], atol=1e-5):
        print(
            f"Reloaded accuracy {metrics['accuracy']:.4f}, "
            f"recorded at fit {record['best_val_accuracy']:.4f}. Check close logits."
        )
    np.save(predictions, scores)
    record.update(
        metrics,
        candidate=name,
        seed=seed,
        checkpoint_sha256=file_digest(checkpoint),
        validation_sha256=file_digest(predictions),
    )
    write_json(record_file, record)
    return record

In [8]:
if MODE != "preview":
    fit_pair, val_pair, official_test_pair, split_id = load_task1_data()
    print(
        f"Official subset: {len(fit_pair[1]):,} fit, {len(val_pair[1]):,} validation, "
        f"{len(official_test_pair[1]):,} test. Split SHA-256: {split_id}"
    )
    settings = experiment_settings(split_id)
    experiment_id = hashlib.sha256(
        json.dumps(settings, sort_keys=True).encode()
    ).hexdigest()[:20]
    directory = (
        TASK1_DIR / ("smoke" if MODE == "smoke" else "experiments") / experiment_id
    )
    directory.mkdir(parents=True, exist_ok=True)
    if MODE == "smoke":
        fit_pair, val_pair = smoke_sample(fit_pair, 2), smoke_sample(val_pair, 1)
        print("Pipeline check only: 100 training and 50 validation images.")
    with task1_writer():
        write_json(directory / "settings.json", settings)
        records = [
            validation_run(name, seed, fit_pair, val_pair, directory)
            for name in CANDIDATES
            for seed in ((42,) if MODE == "smoke" else RUN_SEEDS)
        ]
    run_table = pd.DataFrame(records)
    display(
        run_table[
            [
                "candidate",
                "seed",
                "best_stage",
                "best_epoch",
                "accuracy",
                "macro_f1",
                "top5_accuracy",
                "parameters",
                "seconds",
            ]
        ].round(4)
    )

Official subset: 20,000 fit, 5,000 validation, 5,000 test. Split SHA-256: cef94046a3e1a70ae28edde35213447fbaedd9ee6e5da8f391f96a41bf32ee64
Pipeline check only: 100 training and 50 validation images.


mobilenet_v1 | seed 42 | head | learning rate 0.001


4/4 - 2s - 567ms/step - accuracy: 0.0100 - loss: 5.2675 - val_accuracy: 0.0200 - val_loss: 4.5817


mobilenet_v1 | seed 42 | fine_tune | learning rate 1e-05


4/4 - 3s - 648ms/step - accuracy: 0.0500 - loss: 4.3420 - val_accuracy: 0.0200 - val_loss: 4.5140


custom_cnn | seed 42 | scratch | learning rate 0.001


... (2 outputs left out)

4/4 - 8s - 2s/step - accuracy: 0.0000e+00 - loss: 4.0934 - val_accuracy: 0.0400 - val_loss: 3.9266


efficientnetv2b0 | seed 42 | fine_tune | learning rate 1e-05


4/4 - 11s - 3s/step - accuracy: 0.0300 - loss: 3.7723 - val_accuracy: 0.0400 - val_loss: 3.9197


,candidate,seed,best_stage,best_epoch,accuracy,macro_f1,top5_accuracy,parameters,seconds
0,mobilenet_v1,42,head,1,0.02,0.0200,0.12,3280114,4.9306
1,custom_cnn,42,scratch,1,0.04,0.0034,0.10,294802,2.4046
2,efficientnetv2b0,42,head,1,0.04,0.0333,0.20,5983362,18.9493


### Reading the comparison

The per-seed table uses each run's selected checkpoint. The learning curves show how training and validation accuracy change together, including after unfreezing. Inspect the seed-42 confusion patterns for each architecture to see which classes account for differences in the aggregate scores.

In [9]:
def rank_candidates(table: pd.DataFrame) -> pd.DataFrame:
    expected = {(name, seed) for name in CANDIDATES for seed in RUN_SEEDS}
    received = set(zip(table["candidate"], table["seed"], strict=True))
    if received != expected or len(table) != len(expected):
        raise ValueError("Recommendation requires all nine candidate/seed results.")
    ranking = table.groupby("candidate").agg(
        mean_accuracy=("accuracy", "mean"),
        sd_accuracy=("accuracy", "std"),
        mean_macro_f1=("macro_f1", "mean"),
        mean_top5=("top5_accuracy", "mean"),
        parameters=("parameters", "first"),
        mean_seconds=("seconds", "mean"),
    )
    return ranking.sort_values(
        ["mean_accuracy", "mean_macro_f1", "parameters"], ascending=[False, False, True]
    )


if MODE in {"train", "final"}:
    ranking = rank_candidates(run_table)
    display(ranking.round(4))
    fig, axes = plt.subplots(1, len(CANDIDATES), figsize=(15, 4), sharey=True)
    for ax, name in zip(axes, CANDIDATES, strict=True):
        run = next(r for r in records if r["candidate"] == name and r["seed"] == 42)
        curve = pd.DataFrame(run["history"]).set_index("epoch")
        curve[["accuracy", "val_accuracy"]].plot(ax=ax)
        ax.axvline(run["best_epoch"], color="grey", linestyle=":")
        ax.set(title=name, xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
    fig.tight_layout()
    plt.show()
    for name in CANDIDATES:
        scores = np.load(directory / name / "42" / "validation.npy", allow_pickle=False)
        show_class_errors(val_pair[1], scores, f"{name} | seed 42 | validation")

## Recommendation and final evaluation

Rank the candidates by mean validation accuracy across seeds 42, 43 and 44. Break exact ties by mean macro-F1 and then by fewer parameters. We retain the winning configuration's seed-42 checkpoint. This rule fixes the seed before seeing any validation scores. Compare the winning margin with the across-seed variation and training cost when interpreting the recommendation.

The final run saves one set of test predictions and reuses it when the notebook is reopened. Test scores estimate generalisation to this CIFAR subset. The transfer to new photos is assessed separately in Task 2.

In [10]:
if MODE in {"train", "final"}:
    winner = str(ranking.index[0])
    selected = next(r for r in records if r["candidate"] == winner and r["seed"] == 42)
    margin = ranking.mean_accuracy.iloc[0] - ranking.mean_accuracy.iloc[1]
    print(
        f"Recommend {winner}: mean validation accuracy {ranking.mean_accuracy.iloc[0]:.2%}, "
        f"{100 * margin:.2f} percentage points above the runner-up."
    )
    print(f"Selected checkpoint: {selected['checkpoint']}")

In [11]:
def final_predictions(selection: dict, settings: dict, experiment_id: str, images):
    """Keep the official predictions tied to one selected checkpoint across reruns."""
    with task1_writer():
        final_dir = TASK1_DIR / "final"
        final_dir.mkdir(parents=True, exist_ok=True)
        manifest = final_dir / "selection.json"
        prediction_file = final_dir / "test_scores.npy"
        lock = {
            "experiment_id": experiment_id,
            "candidate": selection["candidate"],
            "seed": 42,
            "checkpoint": selection["checkpoint"],
            "checkpoint_sha256": selection["checkpoint_sha256"],
            "class_names": settings["classes"],
            "original_fine_ids": settings["fine_ids"],
            "input": "RGB 32x32, pixel values 0..255; preprocessing embedded in model",
            "feature_layer": "representation",
            "split_sha256": settings["split_sha256"],
        }
        if manifest.exists():
            if json.loads(manifest.read_text(encoding="utf-8")) != lock:
                raise ValueError(
                    "Official evaluation already belongs to a different selection."
                )
        elif prediction_file.exists():
            raise ValueError("Test predictions are missing their selection manifest.")
        else:
            write_json(manifest, lock)
        if prediction_file.exists():
            return np.load(prediction_file, allow_pickle=False)
        if file_digest(Path(lock["checkpoint"])) != lock["checkpoint_sha256"]:
            raise ValueError("Selected checkpoint changed since validation.")
        model = tf.keras.models.load_model(lock["checkpoint"], compile=False)
        scores = model.predict(images, batch_size=BATCH_SIZE, verbose=0)
        pending = final_dir / "test_scores.pending.npy"
        np.save(pending, scores)
        pending.replace(prediction_file)
        return scores


if MODE == "final":
    test_scores = final_predictions(
        selected, settings, experiment_id, official_test_pair[0]
    )
    test_metrics = classification_metrics(official_test_pair[1], test_scores)
    display(pd.DataFrame([test_metrics], index=[winner]).round(4))
    show_class_errors(official_test_pair[1], test_scores, f"{winner} | official test")
    print(
        "Task 2 checkpoint, feature layer and input contract: artifacts/task1/final/selection.json"
    )

if MODE != "preview":
    tf.keras.mixed_precision.set_global_policy(initial_policy)

## Task 2 — Group-collected image classification

### Collection and labelling

We have selected five container classes: `mug`, `paper_cup`, `plastic_bottle`, `aluminium_can`, and `drink_carton`, in the order recorded in `data/task2/classes.json`. Follow the [collection guide](docs/data-collection.md) for class boundaries and the shooting checklist.

Our collection target is 200 usable photographs per member, with approximately 40 per class. We will coordinate object designs in the shared register, photograph them across two sessions, and review the labels and capture records. Record actual counts, member contributions and exclusions after collection. Use the manifests to create a group-separated training/validation split, keeping shared objects and matching designs together.

### Adaptation and evaluation

Adapt the actual selected Task 1 checkpoint. Explain and evaluate the changes to its head, preprocessing, training stages and augmentation.

### Predictions and error analysis

Display 25 fixed validation examples with actual and predicted labels, then analyse class and condition-specific errors.

## Task 3 — AG News topic classification

Follow [the Task 3 implementation guide](docs/task3-guide.md) for the data audit, model settings and staged PR handoff. Implement the assessed code and retain the essential outputs here.

### Dataset and preprocessing

Load `ag_news_subset:1.0.0` with the pinned TFDS version and combine title with description. Document the original counts and class IDs 0–3. Apply the agreed normalisation, exclude development/test text overlap and conflicting-label development groups, and create the fixed grouped training/validation split. Fit the vocabulary on training data only; report length, unknown-word and truncation statistics.

### Text CNN

Implement one word-level CNN with 128-dimensional embeddings, parallel Conv1D widths 3/4/5, max pooling over valid token windows, dropout and four output logits. Verify padding and checkpoint reload behaviour. Compare 64/128-token inputs using the same split, vocabulary and seed, then select through validation before the final official test evaluation. Include curves, accuracy, macro-F1, per-class metrics, a confusion matrix and real error examples.

### Limitations and improvements

Develop this discussion from the measured results.

## Conclusion

Summarise the findings and trade-offs after all experiments have been completed.

## References

- Huyen, C. (2022). *Designing Machine Learning Systems*. O’Reilly, chapters 4–6.
- [Keras transfer learning and fine-tuning](https://keras.io/guides/transfer_learning/).
- Howard et al. (2017). [MobileNets](https://arxiv.org/abs/1704.04861).
- Tan and Le (2021). [EfficientNetV2](https://proceedings.mlr.press/v139/tan21a.html).
- [TensorFlow Datasets: AG News](https://www.tensorflow.org/datasets/catalog/ag_news_subset).
- Zhang et al. (2015). [Character-level Convolutional Networks for Text Classification](https://arxiv.org/abs/1509.01626).
- Kim (2014). [Convolutional Neural Networks for Sentence Classification](https://aclanthology.org/D14-1181/).